# 02. Model Data Preprocessing

**Goal:** convert the 01 EDA decisions into leakage-safe train and test data.

No model is trained here. No charts are created. All preprocessing rules are learned from the training split only.

## 1. Load and apply confirmed column exclusions

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 100)

DATA_PATH = Path("results.csv")
OUTPUT_DIR = Path("processed")
TARGET = "AISelect"
ID_COLUMN = "ResponseId"
RANDOM_SEED = 42
TEST_SIZE = 0.20
MIN_CATEGORY_COUNT = 200
MIN_MULTI_RATE = 0.01
TARGET_MAPPING = {
    "Yes": 0,
    "No, but I plan to soon": 1,
    "No, and I don't plan to": 2,
}

ADMIRATION_COLUMNS = [
    "LanguageAdmired", "DatabaseAdmired", "PlatformAdmired", "WebframeAdmired",
    "EmbeddedAdmired", "MiscTechAdmired", "ToolsTechAdmired",
    "NEWCollabToolsAdmired", "OfficeStackAsyncAdmired",
    "OfficeStackSyncAdmired", "AISearchDevAdmired",
]
DROP_COLUMNS = ["Check", *ADMIRATION_COLUMNS]
LEAKAGE_COLUMNS = [
    "AISearchDevHaveWorkedWith", "AISearchDevWantToWorkWith", "AISent", "AIComplex",
    "AIToolCurrently Using", "AIToolInterested in Using", "AIToolNot interested in Using",
    "AINextMuch more integrated", "AINextMore integrated", "AINextNo change",
    "AINextLess integrated", "AINextMuch less integrated", "AIThreat", "AIEthics",
    "AIBen", "AIAcc", "AIChallenges",
]
LOW_VALUE_COLUMNS = [
    "SurveyLength", "SurveyEase", "Currency", "CompTotal",
    "JobSatPoints_1", "JobSatPoints_4", "JobSatPoints_5", "JobSatPoints_6",
    "JobSatPoints_7", "JobSatPoints_8", "JobSatPoints_9",
    "JobSatPoints_10", "JobSatPoints_11",
]
WEAK_RELEVANCE_COLUMNS = [
    *[f"Knowledge_{i}" for i in range(1, 10)],
    *[f"Frequency_{i}" for i in range(1, 4)],
    "TimeSearching", "TimeAnswering", "JobSat",
]
MODEL_EXCLUDE_COLUMNS = LEAKAGE_COLUMNS + LOW_VALUE_COLUMNS + WEAK_RELEVANCE_COLUMNS

raw_df = pd.read_csv(DATA_PATH, na_values=["NA", ""], keep_default_na=True, low_memory=False)
df = raw_df.drop(columns=DROP_COLUMNS).copy()
model_df = df.drop(columns=MODEL_EXCLUDE_COLUMNS).copy()
REQUIRED_ROW_COLUMNS = [
    "Age", "Employment", "EdLevel", "Country", "DevType", "YearsCode",
    "WorkExp", "Industry", "ProfessionalTech", "ProfessionalCloud",
]
eligible_rows = (
    model_df[TARGET].notna()
    & model_df["TBranch"].eq("Yes")
    & model_df[REQUIRED_ROW_COLUMNS].notna().all(axis=1)
)
modeling_rows = model_df.loc[eligible_rows].copy()

assert raw_df.shape == (65_437, 114)
assert df.shape[1] == 102
assert model_df.shape[1] == 57
assert len(modeling_rows) == 27_182
row_filter_summary = pd.DataFrame([
    {"Stage": "Raw", "Rows": len(raw_df)},
    {"Stage": "Valid target", "Rows": raw_df[TARGET].notna().sum()},
    {"Stage": "Professional work section", "Rows": (raw_df[TARGET].notna() & raw_df["TBranch"].eq("Yes")).sum()},
    {"Stage": "Required fields complete", "Rows": len(modeling_rows)},
])
row_filter_summary["Removed from previous"] = row_filter_summary["Rows"].shift(1) - row_filter_summary["Rows"]
row_filter_summary["Raw data retained %"] = row_filter_summary["Rows"] / len(raw_df) * 100
display(row_filter_summary)

## 2. Remove direct AI proxy choices

전체 컬럼을 버리지 않고 `AISelect`를 직접 드러내는 선택지만 제거한다. 원래 결측 여부는 이후 별도 지표로 보존한다.

In [ ]:
DIRECT_AI_CHOICES = {
    "LearnCodeOnline": {"AI"},
    "TechDoc": {
        "AI-powered search/dev tool (free)",
        "AI-powered search/dev tool (paid)",
    },
    "BuyNewTool": {"Ask a generative AI tool"},
    "ProfessionalTech": {"AI-assisted technology tool(s)"},
    "ProfessionalQuestion": {"AI-powered search (free)", "AI-powered search (paid)"},
    "TechEndorse": {"AI tool integration"},
}

def remove_choices(value, blocked_choices: set[str]):
    if pd.isna(value):
        return np.nan
    kept = [
        item.strip() for item in str(value).split(";")
        if item.strip() and item.strip() not in blocked_choices
    ]
    return ";".join(kept) if kept else np.nan

clean_df = modeling_rows.copy()
proxy_removal_rows = []
for column, blocked in DIRECT_AI_CHOICES.items():
    before = clean_df[column].notna().sum()
    original = clean_df[column].copy()
    clean_df[column] = clean_df[column].map(lambda value: remove_choices(value, blocked))
    changed = (original.fillna("<NA>") != clean_df[column].fillna("<NA>")).sum()
    proxy_removal_rows.append({
        "Source column": column,
        "Removed choices": " | ".join(sorted(blocked)),
        "Changed rows": changed,
        "Valid before": before,
        "Valid after": clean_df[column].notna().sum(),
    })
proxy_removal_summary = pd.DataFrame(proxy_removal_rows)
display(proxy_removal_summary)

## 3. Reduce numeric redundancy

코딩 경력과 업무 경력은 서로 다른 의미이므로 유지한다. 전문 개발 경력은 업무 경력과 Spearman 0.93으로 중복되어 제외한다.
보수 원본은 로그값으로 교체한다. 이상치는 삭제하지 않는다.

In [ ]:
def to_years(value):
    if pd.isna(value):
        return np.nan
    if value == "Less than 1 year":
        return 0.5
    if value == "More than 50 years":
        return 51.0
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

clean_df["CodingYears"] = clean_df["YearsCode"].map(to_years)
clean_df["WorkExperienceYears"] = pd.to_numeric(clean_df["WorkExp"], errors="coerce")
salary = pd.to_numeric(clean_df["ConvertedCompYearly"], errors="coerce")
clean_df["AnnualCompLog"] = np.log1p(salary.where(salary >= 0))
clean_df = clean_df.drop(columns=[
    "YearsCode", "YearsCodePro", "WorkExp", "ConvertedCompYearly"
]).copy()
constant_source_columns = [
    column for column in clean_df.columns
    if column not in {ID_COLUMN, TARGET} and clean_df[column].nunique(dropna=False) <= 1
]
clean_df = clean_df.drop(columns=constant_source_columns).copy()
print("Constant source columns removed:", constant_source_columns)
print("Numeric features: CodingYears, WorkExperienceYears, AnnualCompLog")

## 4. Stratified train-test split

타겟 비율을 유지해 80:20으로 분리한다. 이후 기준은 학습 데이터에서만 계산한다.

In [ ]:
train_index, test_index = train_test_split(
    clean_df.index, test_size=TEST_SIZE, random_state=RANDOM_SEED,
    stratify=clean_df[TARGET],
)
train_source = clean_df.loc[train_index].copy()
test_source = clean_df.loc[test_index].copy()

split_summary = pd.DataFrame({
    "Train count": train_source[TARGET].value_counts(),
    "Train %": train_source[TARGET].value_counts(normalize=True).mul(100),
    "Test count": test_source[TARGET].value_counts(),
    "Test %": test_source[TARGET].value_counts(normalize=True).mul(100),
})
display(split_summary)

## 5. Detect feature types from training data

In [ ]:
feature_columns = [column for column in clean_df.columns if column not in {ID_COLUMN, TARGET}]
numeric_columns = ["CodingYears", "WorkExperienceYears", "AnnualCompLog"]
object_columns = [column for column in feature_columns if column not in numeric_columns]
multi_columns = [
    column for column in object_columns
    if train_source[column].dropna().astype(str).str.contains(";", regex=False).any()
]
single_columns = [column for column in object_columns if column not in multi_columns]

feature_type_summary = pd.DataFrame({
    "Type": ["Numeric", "Single category", "Multi-select"],
    "Source columns": [len(numeric_columns), len(single_columns), len(multi_columns)],
})
display(feature_type_summary)
print("Multi-select columns:", multi_columns)

## 6. Numeric preprocessing

학습 데이터 중앙값으로 결측을 채우고 결측 여부 컬럼을 추가한다. 스케일링은 모델별 요구가 달라 03에서 처리한다.

In [ ]:
numeric_medians = train_source[numeric_columns].median().to_dict()

def transform_numeric(frame: pd.DataFrame) -> pd.DataFrame:
    output = pd.DataFrame(index=frame.index)
    for column in numeric_columns:
        output[f"num__{column}"] = frame[column].fillna(numeric_medians[column]).astype(float)
        output[f"missing__{column}"] = frame[column].isna().astype("uint8")
    return output

train_numeric = transform_numeric(train_source)
test_numeric = transform_numeric(test_source)
display(pd.DataFrame({"Training median": numeric_medians}))

## 7. Single-category preprocessing

학습 데이터에서 200건 미만 범주를 `Other`로 통합하고 결측은 `Missing`으로 보존한다. 테스트의 새로운 값도 `Other`로 처리한다.

In [ ]:
category_levels = {}
for column in single_columns:
    counts = train_source[column].fillna("Missing").astype(str).value_counts()
    levels = counts[counts >= MIN_CATEGORY_COUNT].index.tolist()
    category_levels[column] = sorted(set(levels + ["Missing", "Other"]))

def transform_single_categories(frame: pd.DataFrame) -> pd.DataFrame:
    parts = []
    for column in single_columns:
        values = frame[column].fillna("Missing").astype(str)
        allowed = set(category_levels[column]) - {"Other"}
        values = values.where(values.isin(allowed), "Other")
        categorical = pd.Categorical(values, categories=category_levels[column])
        encoded = pd.get_dummies(categorical, prefix=f"cat__{column}", dtype="uint8")
        encoded.index = frame.index
        parts.append(encoded)
    return pd.concat(parts, axis=1) if parts else pd.DataFrame(index=frame.index)

train_single = transform_single_categories(train_source)
test_single = transform_single_categories(test_source)
print(f"Single-category encoded features: {train_single.shape[1]}")

## 8. Multi-select preprocessing

학습 응답자의 1% 이상이 선택한 항목만 0/1 변수로 만든다. 원래 결측 여부도 별도로 보존한다.

In [ ]:
minimum_multi_count = max(MIN_CATEGORY_COUNT, int(np.ceil(len(train_source) * MIN_MULTI_RATE)))
multi_vocabulary = {}
for column in multi_columns:
    respondent_sets = train_source[column].fillna("").astype(str).map(
        lambda value: {item.strip() for item in value.split(";") if item.strip()}
    )
    counts = {}
    for choices in respondent_sets:
        for choice in choices:
            counts[choice] = counts.get(choice, 0) + 1
    multi_vocabulary[column] = sorted([
        choice for choice, count in counts.items() if count >= minimum_multi_count
    ])

def transform_multi(frame: pd.DataFrame) -> pd.DataFrame:
    encoded_columns = {}
    for column in multi_columns:
        respondent_sets = frame[column].fillna("").astype(str).map(
            lambda value: {item.strip() for item in value.split(";") if item.strip()}
        )
        encoded_columns[f"missing__{column}"] = frame[column].isna().astype("uint8")
        for choice in multi_vocabulary[column]:
            encoded_columns[f"multi__{column}__{choice}"] = respondent_sets.map(
                lambda values: choice in values
            ).astype("uint8")
    return pd.DataFrame(encoded_columns, index=frame.index)

train_multi = transform_multi(train_source)
test_multi = transform_multi(test_source)
print(f"Minimum multi-select count: {minimum_multi_count}")
print(f"Multi-select encoded features: {train_multi.shape[1]}")

## 9. Combine and validate

PK와 타겟을 다시 붙여 저장용 데이터를 만든다. 모든 입력값은 숫자이고 결측이 없어야 한다.

In [ ]:
X_train = pd.concat([train_numeric, train_single, train_multi], axis=1)
X_test = pd.concat([test_numeric, test_single, test_multi], axis=1)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
constant_encoded_columns = [column for column in X_train.columns if X_train[column].nunique(dropna=False) <= 1]
X_train = X_train.drop(columns=constant_encoded_columns)
X_test = X_test.drop(columns=constant_encoded_columns)

y_train = train_source[TARGET].map(TARGET_MAPPING).astype("int8")
y_test = test_source[TARGET].map(TARGET_MAPPING).astype("int8")

train_processed = pd.concat([
    train_source[[ID_COLUMN]].astype("int64"),
    train_source[[TARGET]].rename(columns={TARGET: "TargetLabel"}),
    y_train.rename("TargetCode"), X_train,
], axis=1)
test_processed = pd.concat([
    test_source[[ID_COLUMN]].astype("int64"),
    test_source[[TARGET]].rename(columns={TARGET: "TargetLabel"}),
    y_test.rename("TargetCode"), X_test,
], axis=1)

assert X_train.columns.equals(X_test.columns)
assert not X_train.isna().any().any()
assert not X_test.isna().any().any()
assert train_processed[ID_COLUMN].is_unique
assert test_processed[ID_COLUMN].is_unique
assert set(train_processed[ID_COLUMN]).isdisjoint(set(test_processed[ID_COLUMN]))
assert all(pd.api.types.is_numeric_dtype(X_train[column]) for column in X_train.columns)

validation_summary = pd.DataFrame([
    {"Split": "Train", "Rows": len(train_processed), "Input features": X_train.shape[1],
     "Missing input values": int(X_train.isna().sum().sum())},
    {"Split": "Test", "Rows": len(test_processed), "Input features": X_test.shape[1],
     "Missing input values": int(X_test.isna().sum().sum())},
])
display(validation_summary)

## 10. Save processed data and metadata

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
train_path = OUTPUT_DIR / "ai_train.csv.gz"
test_path = OUTPUT_DIR / "ai_test.csv.gz"
metadata_path = OUTPUT_DIR / "preprocessing_metadata.json"

train_processed.to_csv(train_path, index=False, compression="gzip")
test_processed.to_csv(test_path, index=False, compression="gzip")

metadata = {
    "source": str(DATA_PATH),
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "target_mapping": TARGET_MAPPING,
    "train_rows": len(train_processed),
    "test_rows": len(test_processed),
    "input_feature_count": X_train.shape[1],
    "required_row_columns": REQUIRED_ROW_COLUMNS,
    "eligible_rule": "AISelect present, TBranch Yes, required fields complete",
    "dropped_duplicate_numeric": ["YearsCodePro"],
    "constant_source_columns_removed": constant_source_columns,
    "constant_encoded_columns_removed": constant_encoded_columns,
    "numeric_features": numeric_columns,
    "numeric_medians": numeric_medians,
    "single_category_columns": single_columns,
    "multi_select_columns": multi_columns,
    "minimum_category_count": MIN_CATEGORY_COUNT,
    "minimum_multi_count": minimum_multi_count,
    "direct_ai_choices_removed": {k: sorted(v) for k, v in DIRECT_AI_CHOICES.items()},
    "category_levels": category_levels,
    "multi_vocabulary": multi_vocabulary,
    "input_columns": X_train.columns.tolist(),
}
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

for path in [train_path, test_path, metadata_path]:
    print(f"Saved: {path} ({path.stat().st_size / 1024**2:.2f} MB)")

## Output

- `processed/ai_train.csv.gz`: training data
- `processed/ai_test.csv.gz`: untouched test data
- `processed/preprocessing_metadata.json`: rules learned from training data

03은 이 파일을 읽어 모델을 비교한다. 테스트 데이터는 최종 평가 전까지 학습에 사용하지 않는다.